# Partie III — RNN, LSTM, GRU et Seq2Seq
### Modélisation de séquences et traduction Anglais → Français — corpus *fra-eng* (Tatoeba)

**Projet de fin de module — Deep Learning — EMSI Casablanca, 2025–2026**

---

Notebook **autonome** : import des données, configuration, théorie,
implémentations, expériences et analyse y figurent. **Rien n'est codé en dur** :
tout passe par la cellule `Config`.

**Plan**
1. Modèle de langage : objectif probabiliste, règle de chaîne, perplexité
2. Préparation des données (tokenisation, vocabulaire, tokens spéciaux, padding, masquage, mini-lots)
3. RNN / LSTM / GRU : implémentation et comparaison (stabilité, performance, mémoire, coût)
4. BPTT et effet du *gradient clipping* (expérience)
5. Système Seq2Seq encodeur–décodeur (teacher forcing, perte masquée)
6. Décodage : glouton et *beam search*
7. Évaluation (perplexité, BLEU) et analyse critique


## 1. Modèle de langage : objectif probabiliste et perplexité

Un **modèle de langage** attribue une probabilité à une suite de tokens
$P(x_1,\dots,x_T)$. Par la **règle de chaîne** :
$$P(x_1,\dots,x_T)=\prod_{t=1}^{T} P(x_t \mid x_1,\dots,x_{t-1}).$$
Modéliser une phrase entière revient donc à savoir **prédire le prochain token**
à partir du passé ; toute la difficulté est de *représenter ce passé*. Un RNN
résume ce passé dans un **état caché** $h_t$ transmis de pas en pas :
$$h_t=\phi(W_{xh}x_t+W_{hh}h_{t-1}+b_h),\qquad \hat y_t=\mathrm{softmax}(W_{hq}h_t+b_q).$$

**Perplexité.** Mesure standard de qualité d'un modèle de langage :
$$\mathrm{PPL}=\exp\!\left(-\frac{1}{T}\sum_{t=1}^{T}\log P(x_t\mid x_{<t})\right).$$
C'est l'exponentielle de l'entropie croisée moyenne. Interprétation : le nombre
moyen de choix « équiprobables » entre lesquels le modèle hésite à chaque pas.
**Plus la perplexité est faible, meilleur est le modèle** (PPL = 1 = parfait ;
PPL = $|V|$ = aléatoire uniforme).


In [ ]:
import os, sys, subprocess, random, json, glob, re, math, time
from dataclasses import dataclass, field, asdict
from typing import List

def _ensure(pkg, imp=None):
    import importlib
    try: importlib.import_module(imp or pkg)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
for _p, _i in [("numpy","numpy"),("matplotlib","matplotlib"),("seaborn","seaborn"),
               ("torch","torch"),("kagglehub","kagglehub")]:
    _ensure(_p, _i)

import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from torch.nn import functional as F
sns.set_theme(style="whitegrid")

@dataclass
class Config:
    seed: int = 42
    max_pairs: int = 8000        # nb de paires phrase utilisées (CPU-friendly)
    max_len: int = 9             # longueur max (tokens) hors <bos>/<eos>
    min_freq: int = 2            # fréquence min pour entrer au vocabulaire
    val_frac: float = 0.1
    # Modèle
    embed_dim: int = 96
    hidden_dim: int = 128
    # Optimisation
    batch_size: int = 128
    lr: float = 3e-3
    epochs_lm: int = 6           # modèles de langage RNN/LSTM/GRU
    epochs_seq2seq: int = 12
    clip_norm: float = 1.0
    # Décodage
    beam_width: int = 3
    kaggle_id: str = "myksust/fra-eng"
    data_dir: str = os.path.join("..", "data", "fra_eng")
    artifacts_dir: str = os.path.join("..", "artifacts", "part3_rnn")

CFG = Config()
os.makedirs(CFG.data_dir, exist_ok=True); os.makedirs(CFG.artifacts_dir, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def try_gpu(i=0):
    return torch.device(f"cuda:{i}") if torch.cuda.device_count() >= i+1 else torch.device("cpu")
set_seed(CFG.seed); DEVICE = try_gpu()
print("Device :", DEVICE); print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))

## 2. Préparation des données

Le corpus *fra-eng* (Tatoeba) est un fichier texte de paires `anglais\tfrançais`.
Pipeline : nettoyage → tokenisation → vocabulaire avec **tokens spéciaux**
(`<pad>`, `<bos>`, `<eos>`, `<unk>`) → troncature/padding → mini-lots tensoriels.

In [ ]:
import urllib.request, zipfile

FALLBACK_URL = "https://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip"  # corpus Tatoeba, sans auth

def list_data_files(root):
    files = []
    for ext in ("*.txt", "*.csv", "*.tsv"):
        files += glob.glob(os.path.join(root, "**", ext), recursive=True)
    return files

def _read_lines(path):
    """Lecture robuste : essaie plusieurs encodages courants."""
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            with open(path, encoding=enc) as f:
                return f.read().splitlines()
        except UnicodeDecodeError:
            continue
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read().splitlines()

def _parse_pairs(lines):
    """Detecte le separateur (tabulation ou virgule) et extrait les paires."""
    pairs = []
    for line in lines:
        if not line.strip():
            continue
        parts = line.split("\t") if "\t" in line else line.split(",")
        if len(parts) >= 2 and parts[0].strip() and parts[1].strip():
            pairs.append((parts[0].strip(), parts[1].strip()))
    return pairs

def _download_fallback(cfg):
    """Repli public (sans Kaggle) : telecharge et extrait le corpus Tatoeba fra-eng."""
    os.makedirs(cfg.data_dir, exist_ok=True)
    if not list_data_files(cfg.data_dir):
        zip_path = os.path.join(cfg.data_dir, "fra-eng.zip")
        print("Telechargement du corpus depuis un miroir public (sans Kaggle)...")
        urllib.request.urlretrieve(FALLBACK_URL, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(cfg.data_dir)
    return cfg.data_dir

def ensure_corpus(cfg):
    # 1) fichiers deja presents en local
    if list_data_files(cfg.data_dir):
        return cfg.data_dir
    # 2) Kaggle (uniquement si authentifie ; sinon on bascule sur le repli)
    try:
        import kagglehub
        root = kagglehub.dataset_download(cfg.kaggle_id)
        if list_data_files(root):
            return root
    except Exception as e:
        print(f"Kaggle indisponible ({type(e).__name__}) -> repli sur le miroir public.")
    # 3) repli public sans authentification
    return _download_fallback(cfg)

def load_pairs(cfg):
    root = ensure_corpus(cfg)
    files = list_data_files(root)
    print("Fichiers trouves :")
    for fp in files:
        print("  -", fp)

    # on retient le fichier qui produit le PLUS de paires valides
    best_path, best_pairs = None, []
    for fp in files:
        cand = _parse_pairs(_read_lines(fp))
        if len(cand) > len(best_pairs):
            best_path, best_pairs = fp, cand
    if not best_pairs:
        raise ValueError("Corpus trouve mais aucune paire 'source<sep>cible' detectee.")

    # retire une eventuelle ligne d'en-tete
    if best_pairs[0][0].lower() in ("english", "eng", "en", "source", "src"):
        best_pairs = best_pairs[1:]

    print("\nFichier corpus retenu :", best_path, "| paires valides :", len(best_pairs))
    return best_pairs

raw_pairs = load_pairs(CFG)
print("Paires brutes :", len(raw_pairs))
print("Exemple :", raw_pairs[min(100, len(raw_pairs) - 1)])   # index sur

In [ ]:
_token_re = re.compile(r"[a-zà-ÿ0-9]+|[?.!,;:']", re.IGNORECASE)
def tokenize(text):
    return _token_re.findall(text.lower())

# Filtrage : on garde les paires courtes (entraînement raisonnable sur CPU)
pairs = []
for en, fr in raw_pairs:
    te, tf = tokenize(en), tokenize(fr)
    if 1 <= len(te) <= CFG.max_len and 1 <= len(tf) <= CFG.max_len:
        pairs.append((te, tf))
random.Random(CFG.seed).shuffle(pairs)
pairs = pairs[:CFG.max_pairs]
print("Paires retenues :", len(pairs))
print("Exemple tokenisé :", pairs[0])

In [ ]:
from collections import Counter

PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"

class Vocab:
    def __init__(self, token_lists, min_freq=1):
        counter = Counter(tok for toks in token_lists for tok in toks)
        self.itos = [PAD, BOS, EOS, UNK] + \
                    [t for t, c in counter.most_common() if c >= min_freq]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.pad, self.bos, self.eos, self.unk = (self.stoi[PAD], self.stoi[BOS],
                                                  self.stoi[EOS], self.stoi[UNK])
    def __len__(self): return len(self.itos)
    def encode(self, toks): return [self.stoi.get(t, self.unk) for t in toks]
    def decode(self, ids):
        out = []
        for i in ids:
            tok = self.itos[i]
            if tok == EOS: break
            if tok not in (PAD, BOS): out.append(tok)
        return out

src_sents = [en for en, fr in pairs]   # anglais (source)
tgt_sents = [fr for en, fr in pairs]   # français (cible)
vocab_src = Vocab(src_sents, CFG.min_freq)
vocab_tgt = Vocab(tgt_sents, CFG.min_freq)
print(f"Vocabulaire source (EN) : {len(vocab_src)} | cible (FR) : {len(vocab_tgt)}")

In [ ]:
def pad_to(ids, length, pad_id):
    return ids[:length] + [pad_id] * max(0, length - len(ids))

def make_lm_tensors(sentences, vocab, max_len):
    """Modèle de langage : seq = <bos> tokens <eos> ; entrée=seq[:-1], cible=seq[1:]."""
    L = max_len + 2
    X, Y = [], []
    for toks in sentences:
        seq = [vocab.bos] + vocab.encode(toks) + [vocab.eos]
        seq = pad_to(seq, L, vocab.pad)
        X.append(seq[:-1]); Y.append(seq[1:])
    return torch.tensor(X), torch.tensor(Y)

# Modèle de langage construit sur la langue cible (français)
X_lm, Y_lm = make_lm_tensors(tgt_sents, vocab_tgt, CFG.max_len)
n_val = int(len(X_lm) * CFG.val_frac)
lm_train = (X_lm[n_val:], Y_lm[n_val:])
lm_val   = (X_lm[:n_val], Y_lm[:n_val])
print("LM entrée :", X_lm.shape, "| cible :", Y_lm.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
def tensor_loader(X, Y, bs, shuffle):
    return DataLoader(TensorDataset(X, Y), batch_size=bs, shuffle=shuffle)
lm_train_loader = tensor_loader(*lm_train, CFG.batch_size, True)
lm_val_loader   = tensor_loader(*lm_val,   CFG.batch_size, False)

## 3. RNN, LSTM, GRU : implémentation et comparaison

On encapsule les trois cellules récurrentes derrière une **même interface** de
modèle de langage, pilotée par le paramètre `cell`. La perte masquée ignore les
positions `<pad>` (`ignore_index`).

- **RNN simple** : léger mais sujet aux gradients évanescents/explosifs sur
  longues dépendances.
- **LSTM** : état de cellule $c_t$ + 3 portes → chemin de gradient stable.
- **GRU** : 2 portes, pas d'état de cellule séparé → bon compromis coût/perf.

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, cell="rnn", pad_idx=0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.cell = cell
    def forward(self, x):
        out, _ = self.rnn(self.embed(x))
        return self.fc(out)                      # (B, T, V)

def masked_ce(logits, target, pad_idx):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                           target.reshape(-1), ignore_index=pad_idx)

def perplexity(model, loader, pad_idx, device):
    model.eval(); tot_loss = tot_tok = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            ntok = (yb != pad_idx).sum().item()
            tot_loss += masked_ce(logits, yb, pad_idx).item() * ntok
            tot_tok += ntok
    return math.exp(tot_loss / max(tot_tok, 1))

In [ ]:
def train_lm(model, train_loader, val_loader, cfg, device, clip=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    pad = vocab_tgt.pad
    hist = {"val_ppl": [], "epoch_time": []}
    for ep in range(cfg.epochs_lm):
        model.train(); t0 = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = masked_ce(model(xb), yb, pad)
            loss.backward()
            if clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        hist["epoch_time"].append(time.time() - t0)
        hist["val_ppl"].append(perplexity(model, val_loader, pad, device))
    return hist

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

lm_results = {}
for cell in ["rnn", "lstm", "gru"]:
    set_seed(CFG.seed)
    m = RNNLanguageModel(len(vocab_tgt), CFG.embed_dim, CFG.hidden_dim, cell, vocab_tgt.pad)
    h = train_lm(m, lm_train_loader, lm_val_loader, CFG, DEVICE, clip=CFG.clip_norm)
    lm_results[cell] = {"hist": h, "params": count_params(m),
                        "final_ppl": h["val_ppl"][-1],
                        "avg_time": float(np.mean(h["epoch_time"]))}
    print(f"{cell.upper():5s} | PPL finale={h['val_ppl'][-1]:.2f} | "
          f"params={count_params(m):,} | temps/époque={np.mean(h['epoch_time']):.2f}s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for cell, r in lm_results.items():
    ax[0].plot(range(1, CFG.epochs_lm + 1), r["hist"]["val_ppl"], marker="o", label=cell.upper())
ax[0].set_title("Perplexité de validation"); ax[0].set_xlabel("époque")
ax[0].set_ylabel("PPL (plus bas = mieux)"); ax[0].legend()

import pandas as pd
summary = pd.DataFrame({c: {"PPL finale": r["final_ppl"], "params": r["params"],
                            "temps/époque (s)": round(r["avg_time"], 2)}
                        for c, r in lm_results.items()}).T
ax[1].axis("off")
ax[1].table(cellText=np.round(summary.values, 2), colLabels=summary.columns,
            rowLabels=[c.upper() for c in summary.index], loc="center")
ax[1].set_title("Comparaison RNN / LSTM / GRU")
plt.tight_layout(); plt.show()
summary

**Interprétation.** LSTM et GRU atteignent une **perplexité plus basse** que le
RNN simple : leurs portes stabilisent l'apprentissage et capturent mieux le
contexte. Le RNN simple est le **plus léger et le plus rapide** par époque mais
le moins performant. Le GRU offre généralement le **meilleur compromis**
(proche du LSTM, moins de paramètres). Ces observations confirment la théorie
des sections suivantes.

## 4. BPTT et effet du *gradient clipping*

**BPTT (Backpropagation Through Time).** On *déplie* le RNN dans le temps : le
gradient d'une perte finale par rapport à un état antérieur contient une chaîne
de produits jacobiens
$\frac{\partial L}{\partial h_k}=\sum_{t\ge k}\frac{\partial L}{\partial h_t}\prod_{i=k+1}^{t}\frac{\partial h_i}{\partial h_{i-1}}$.
Ce produit explique les gradients **évanescents** (→ 0) ou **explosifs** (→ ∞).
Le **gradient clipping** renormalise le gradient si sa norme dépasse un seuil
$\theta$ : $g \leftarrow \min(1, \theta/\lVert g\rVert)\,g$.

**Expérience.** On entraîne un RNN simple avec un **learning rate élevé** (pour
provoquer l'explosion) et on enregistre la **norme du gradient** par pas, *avec*
et *sans* clipping.

In [ ]:
def track_grad_norms(clip, lr=0.05, steps=120):
    set_seed(CFG.seed)
    model = RNNLanguageModel(len(vocab_tgt), CFG.embed_dim, CFG.hidden_dim, "rnn", vocab_tgt.pad).to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr)   # SGD + LR élevé = instable
    norms, it = [], iter(lm_train_loader)
    for _ in range(steps):
        try: xb, yb = next(it)
        except StopIteration: it = iter(lm_train_loader); xb, yb = next(it)
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        masked_ce(model(xb), yb, vocab_tgt.pad).backward()
        total = math.sqrt(sum(p.grad.pow(2).sum().item() for p in model.parameters() if p.grad is not None))
        norms.append(total)
        if clip is not None:
            nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
    return norms

norms_noclip = track_grad_norms(clip=None)
norms_clip   = track_grad_norms(clip=CFG.clip_norm)

plt.figure(figsize=(10, 4))
plt.plot(norms_noclip, label="sans clipping", alpha=0.8)
plt.plot(norms_clip, label=f"avec clipping (θ={CFG.clip_norm})", alpha=0.8)
plt.yscale("log"); plt.xlabel("itération"); plt.ylabel("norme du gradient (échelle log)")
plt.title("Effet du gradient clipping sur la stabilité de la BPTT"); plt.legend()
plt.tight_layout(); plt.show()
print(f"Norme max sans clipping : {max(norms_noclip):.1f} | avec clipping : {max(norms_clip):.1f}")

**Lecture.** Sans clipping, la norme du gradient présente des **pics élevés**
(explosion) qui déstabilisent l'apprentissage. Le clipping la **plafonne**, ce
qui produit une descente bien plus régulière — illustration directe de
l'intérêt du clipping pour entraîner des RNN.

## 5. Système Seq2Seq encodeur–décodeur

On construit un mini système de traduction EN→FR : un **encodeur GRU** comprime
la phrase source en un état, un **décodeur GRU** génère la cible token par token.
Entraînement par **teacher forcing** (on fournit le vrai token précédent) et
**perte masquée** (les `<pad>` sont ignorés).

In [ ]:
def make_seq2seq_tensors(pairs, vsrc, vtgt, max_len):
    L = max_len + 2
    SRC, TIN, TOUT = [], [], []
    for te, tf in pairs:
        s = pad_to(vsrc.encode(te) + [vsrc.eos], L, vsrc.pad)
        tgt = [vtgt.bos] + vtgt.encode(tf) + [vtgt.eos]
        tgt = pad_to(tgt, L, vtgt.pad)
        SRC.append(s); TIN.append(tgt[:-1]); TOUT.append(tgt[1:])
    return torch.tensor(SRC), torch.tensor(TIN), torch.tensor(TOUT)

SRC, TIN, TOUT = make_seq2seq_tensors(pairs, vocab_src, vocab_tgt, CFG.max_len)
n_val = int(len(SRC) * CFG.val_frac)
s2s_train = DataLoader(TensorDataset(SRC[n_val:], TIN[n_val:], TOUT[n_val:]),
                       batch_size=CFG.batch_size, shuffle=True)
s2s_val   = DataLoader(TensorDataset(SRC[:n_val], TIN[:n_val], TOUT[:n_val]),
                       batch_size=CFG.batch_size, shuffle=False)
val_pairs = pairs[:n_val]
print("Seq2Seq — source:", SRC.shape, "| tgt_in:", TIN.shape, "| tgt_out:", TOUT.shape)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
    def forward(self, src):
        out, h = self.gru(self.embed(src))
        return out, h                            # h : (1, B, H) état final

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, tgt_in, hidden):
        out, h = self.gru(self.embed(tgt_in), hidden)
        return self.fc(out), h

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__(); self.encoder = encoder; self.decoder = decoder
    def forward(self, src, tgt_in):
        _, state = self.encoder(src)
        logits, _ = self.decoder(tgt_in, state)
        return logits

set_seed(CFG.seed)
enc = Encoder(len(vocab_src), CFG.embed_dim, CFG.hidden_dim, vocab_src.pad)
dec = Decoder(len(vocab_tgt), CFG.embed_dim, CFG.hidden_dim, vocab_tgt.pad)
seq2seq = Seq2Seq(enc, dec).to(DEVICE)
print("Paramètres Seq2Seq :", count_params(seq2seq))

In [ ]:
def train_seq2seq(model, train_loader, val_loader, cfg, device):
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    pad = vocab_tgt.pad; hist = {"train_loss": [], "val_loss": []}
    for ep in range(cfg.epochs_seq2seq):
        model.train(); run = ntok = 0
        for src, tin, tout in train_loader:
            src, tin, tout = src.to(device), tin.to(device), tout.to(device)
            opt.zero_grad()
            loss = masked_ce(model(src, tin), tout, pad)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_norm)   # clipping
            opt.step()
            n = (tout != pad).sum().item(); run += loss.item() * n; ntok += n
        tr = run / ntok
        # validation
        model.eval(); vr = vt = 0
        with torch.no_grad():
            for src, tin, tout in val_loader:
                src, tin, tout = src.to(device), tin.to(device), tout.to(device)
                n = (tout != pad).sum().item()
                vr += masked_ce(model(src, tin), tout, pad).item() * n; vt += n
        hist["train_loss"].append(tr); hist["val_loss"].append(vr / vt)
        if (ep + 1) % 2 == 0:
            print(f"ép {ep+1:2d} | train {tr:.3f} | val {vr/vt:.3f} | val PPL {math.exp(vr/vt):.2f}")
    return hist

hist_s2s = train_seq2seq(seq2seq, s2s_train, s2s_val, CFG, DEVICE)
torch.save(seq2seq.state_dict(), os.path.join(CFG.artifacts_dir, "seq2seq_gru.pt"))

## 6. Décodage : glouton et *beam search*

À l'inférence, pas de vrai token précédent : on génère **auto-régressivement**.
- **Décodage glouton** : à chaque pas, on prend l'argmax. Rapide, mais
  localement optimal seulement.
- **Beam search** : on conserve les $k$ meilleures hypothèses partielles (score
  = somme des log-probabilités), avec **pénalité de longueur** pour ne pas
  favoriser les séquences trop courtes.

In [ ]:
@torch.no_grad()
def greedy_decode(model, src_ids, max_len, device):
    model.eval()
    src = torch.tensor([src_ids], device=device)
    _, state = model.encoder(src)
    tok = torch.tensor([[vocab_tgt.bos]], device=device)
    out = []
    for _ in range(max_len):
        logits, state = model.decoder(tok, state)
        nxt = int(logits[0, -1].argmax())
        if nxt == vocab_tgt.eos: break
        out.append(nxt); tok = torch.tensor([[nxt]], device=device)
    return vocab_tgt.decode(out)

@torch.no_grad()
def beam_search_decode(model, src_ids, max_len, device, beam_width=3, alpha=0.7):
    model.eval()
    src = torch.tensor([src_ids], device=device)
    _, state = model.encoder(src)
    # chaque hypothèse : (log_prob, tokens, state, finie?)
    beams = [(0.0, [vocab_tgt.bos], state, False)]
    for _ in range(max_len):
        candidates = []
        for logp, toks, st, done in beams:
            if done:
                candidates.append((logp, toks, st, True)); continue
            tok = torch.tensor([[toks[-1]]], device=device)
            logits, st2 = model.decoder(tok, st)
            logprobs = F.log_softmax(logits[0, -1], dim=-1)
            topv, topi = logprobs.topk(beam_width)
            for v, i in zip(topv.tolist(), topi.tolist()):
                fin = (i == vocab_tgt.eos)
                candidates.append((logp + v, toks + [i], st2, fin))
        # pénalité de longueur : score normalisé
        candidates.sort(key=lambda c: c[0] / (len(c[1]) ** alpha), reverse=True)
        beams = candidates[:beam_width]
        if all(b[3] for b in beams): break
    best = max(beams, key=lambda c: c[0] / (len(c[1]) ** alpha))
    return vocab_tgt.decode(best[1][1:])     # retire <bos>

In [ ]:
# Exemples qualitatifs de traduction
print("Quelques traductions (source EN -> référence FR | glouton | beam) :\n")
for te, tf in val_pairs[:8]:
    src_ids = pad_to(vocab_src.encode(te) + [vocab_src.eos], CFG.max_len + 2, vocab_src.pad)
    g = " ".join(greedy_decode(seq2seq, src_ids, CFG.max_len + 2, DEVICE))
    b = " ".join(beam_search_decode(seq2seq, src_ids, CFG.max_len + 2, DEVICE, CFG.beam_width))
    print(f"EN : {' '.join(te)}")
    print(f"REF: {' '.join(tf)}")
    print(f"GLO: {g}")
    print(f"BEA: {b}\n")

## 7. Évaluation (BLEU) et analyse critique

Le score **BLEU** mesure la qualité de traduction via la précision des
$n$-grammes, pondérée par une **pénalité de brièveté**. On l'implémente
« maison » pour bien comprendre la métrique, puis on compare glouton vs beam.

In [ ]:
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def sentence_bleu(candidate, reference, max_n=4):
    if len(candidate) == 0: return 0.0
    weights = [1.0 / max_n] * max_n
    log_prec = 0.0
    for n in range(1, max_n + 1):
        cand_ng = ngram_counts(candidate, n)
        ref_ng = ngram_counts(reference, n)
        overlap = sum(min(c, ref_ng.get(g, 0)) for g, c in cand_ng.items())
        total = max(sum(cand_ng.values()), 1)
        # lissage léger pour éviter log(0)
        prec = (overlap + 1e-9) / total
        log_prec += weights[n - 1] * math.log(prec)
    bp = 1.0 if len(candidate) > len(reference) else math.exp(1 - len(reference) / max(len(candidate), 1))
    return bp * math.exp(log_prec)

def corpus_bleu(decoder_fn, pairs, n_eval=400):
    scores = []
    for te, tf in pairs[:n_eval]:
        src_ids = pad_to(vocab_src.encode(te) + [vocab_src.eos], CFG.max_len + 2, vocab_src.pad)
        cand = decoder_fn(src_ids)
        scores.append(sentence_bleu(cand, tf))
    return float(np.mean(scores))

bleu_greedy = corpus_bleu(lambda s: greedy_decode(seq2seq, s, CFG.max_len + 2, DEVICE), val_pairs)
bleu_beam   = corpus_bleu(lambda s: beam_search_decode(seq2seq, s, CFG.max_len + 2, DEVICE, CFG.beam_width), val_pairs)
print(f"BLEU (validation) — glouton : {bleu_greedy:.4f}")
print(f"BLEU (validation) — beam    : {bleu_beam:.4f}  (k={CFG.beam_width})")

### Analyse critique

- **RNN vs LSTM/GRU.** L'expérience de perplexité (section 3) montre que les
  cellules à portes (LSTM/GRU) battent le RNN simple : elles atténuent le
  problème des gradients évanescents et mémorisent mieux le contexte. Le RNN
  simple reste le plus rapide/léger mais le moins performant.
- **Gradient clipping (section 4).** Sans lui, la norme du gradient explose et
  l'entraînement diverge ; le clipping rend la BPTT stable — indispensable pour
  les RNN.
- **Seq2Seq.** Le modèle apprend des traductions correctes pour les phrases
  courtes et fréquentes ; il échoue sur les phrases rares/longues car
  **tout le contexte source est compressé dans un seul vecteur** (goulot
  d'étranglement), faute de mécanisme d'**attention**.
- **Glouton vs beam.** Le beam search, en explorant plusieurs hypothèses et avec
  pénalité de longueur, donne en général un BLEU **égal ou supérieur** au
  glouton, au prix d'un coût de calcul plus élevé.
- **Limites pratiques.** Corpus et budget réduits (portabilité CPU) ; un BLEU
  modeste est attendu. Augmenter `max_pairs`, `epochs_seq2seq`, `hidden_dim`
  dans `Config`, ou ajouter de l'attention, améliorerait nettement les scores.

### Question de synthèse — Partie III
> *Dans quelle mesure les architectures récurrentes modélisent-elles
> efficacement une séquence réelle, et comment justifier le passage d'un RNN
> simple vers un LSTM/GRU puis vers un schéma encodeur–décodeur ?*

Les architectures récurrentes modélisent une séquence en factorisant sa
probabilité par la **règle de chaîne** et en résumant le passé dans un **état
caché** transmis de pas en pas, ce qui gère naturellement des longueurs
variables et l'ordre des tokens — d'où une perplexité bien inférieure aux
$n$-grammes. Toutefois, le **RNN simple** souffre de la BPTT : la chaîne de
jacobiens provoque gradients évanescents/explosifs (atténués par le clipping,
cf. section 4), limitant la portée mémorielle. Le passage au **LSTM/GRU** se
justifie par leurs **portes** et, pour le LSTM, un **état de cellule** offrant
un chemin de gradient stable : nos mesures de perplexité confirment un gain net
de performance et de stabilité, le GRU étant un bon compromis coût/précision.
Enfin, le schéma **encodeur–décodeur** se justifie dès que l'entrée et la sortie
sont deux séquences de longueurs **différentes et non alignées** (traduction) :
l'encodeur produit une représentation conditionnante, le décodeur génère la
cible de façon auto-régressive (teacher forcing à l'entraînement, glouton/beam à
l'inférence, évaluation BLEU). Sa **limite** — la compression de tout le contexte
en un vecteur — motive historiquement l'introduction de l'**attention**, étape
suivante naturelle de cette progression.
